# Instituto Tecnológico y de Estudios Superiores de Monterrey

## Análisis de Grandes Volúmenes de Datos

### Actividad 3 – Aprendizaje supervisado y no supervisado

**Profesor:** Dr. Iván Olmos Pineda  
**Fecha de entrega:** 30 de mayo de 2026  
**Alumno:** Hiram García Austria  
**Matricula:** A0038771

---

## Objetivo de la actividad

Aplicar algoritmos de aprendizaje supervisado y no supervisado mediante PySpark para la resolución de problemas en análisis de datos, fomentando el desarrollo de habilidades prácticas en el manejo y procesamiento eficiente de grandes conjuntos de datos.

---

## Contexto del proyecto

Como parte del proyecto del curso, se seleccionó un dataset histórico del mercado bursátil que contiene información de más de 9,000 acciones con registros diarios desde 1962.

Este conjunto de datos incluye variables financieras clave como precios de apertura, cierre, máximos, mínimos, volumen de transacciones, dividendos y ajustes por división de acciones, representando más de 34 millones de registros y un tamaño aproximado de 4.47 GB.

La población objetivo corresponde al universo completo de registros históricos contenidos en el dataset bursátil seleccionado.

---

## 1. Introducción teórica: Tipos de Aprendizaje Automático

**Aprendizaje Supervisado**
El aprendizaje supervisado utiliza datos que ya tienen una respuesta conocida (etiqueta). El objetivo es que el modelo aprenda la relación entre las variables de entrada y la respuesta para poder hacer predicciones sobre nuevos datos.

Los algoritmos más representativos son los siguientes (todos disponibles en PySpark MLlib):
- Regresión Lineal
- Regresión Logística
- Árboles de Decisión
- Bosques Aleatorios
- Máquinas de Soporte Vectorial (SVM)
- Naive Bayes
- Gradient Boosting
- Redes Neuronales

**Aprendizaje No Supervisado**
El aprendizaje no supervisado trabaja con datos que no tienen etiquetas. Su objetivo es encontrar patrones, grupos o estructuras ocultas dentro de los datos.

Los algoritmos más representativos son:
- K-Means
- Clustering Jerárquico
- Gaussian Mixture Models
- DBSCAN
- Análisis de Componentes Principales
- Análisis de Reglas de Asociación

**Nota:** Algoritmos como DBSCAN y Clustering Jerárquico no están incluidos de forma nativa en PySpark MLlib y suelen requerir librerías externas o implementaciones personalizadas.

---

## 2. Selección de los datos

A continuación se recolecta una muestra de dimensión contenida de la base de datos (D). Para ello se obtienen particiones que cumplan con los criterios de las variables de caracterización identificadas:
* Periodo económico  
* Volumen de transacciones
* Nivel de volatilidad

Después se obtendrán un número limitado de instancias de cada partición aplicando la técnica de **muestreo estratificado**, lo que permitirá construir una muestra (M) a partir de la unión de las instancias que se recuperan de este proceso.

In [28]:
import os
import findspark
import kagglehub

findspark.init()
from pyspark import SparkContext, SparkConf, SQLContext
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import expr, count, when, col, to_date, lit, round, concat_ws

spark = SparkSession.builder \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

# Property used to format output tables better spark
spark.conf.set("spark.sql.repl.eagerEval.enabled", True)

In [29]:
# Download latest version
path = kagglehub.dataset_download("jakewright/9000-tickers-of-stock-market-data-full-history")

print("Path to dataset files:", path)

Path to dataset files: C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2


In [30]:
path = path + "/all_stock_data.csv"
print(path)

C:\Users\hille\.cache\kagglehub\datasets\jakewright\9000-tickers-of-stock-market-data-full-history\versions\2/all_stock_data.csv


In [31]:
# Definimos el esquema del data set
schema = StructType([
    StructField("Date", DateType(), True),
    StructField("Ticker", StringType(), True),
    StructField("Open", DoubleType(), True),
    StructField("High", DoubleType(), True),
    StructField("Low", DoubleType(), True),
    StructField("Close", DoubleType(), True),
    StructField("Volume", DoubleType(), True),
    StructField("Dividends", DecimalType(5,1), True),
    StructField("Stock Splits", DecimalType(5,1), True)
])

In [32]:
# Disparador 0
# Leemos el data set e imprimimos los primeros 5 registros
df = spark.read.csv(path, header=True, inferSchema=False, schema=schema)
df.show(5)

+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|      Date|Ticker|Open|               High|                Low|              Close|   Volume|Dividends|Stock Splits|
+----------+------+----+-------------------+-------------------+-------------------+---------+---------+------------+
|1962-01-02|    ED| 0.0| 0.2658275556233194|0.26178762316703796|0.26178762316703796|  25600.0|      0.0|         0.0|
|1962-01-02|   CVX| 0.0|0.04680890217423439|0.04606926600933256|0.04680890217423439| 105840.0|      0.0|         0.0|
|1962-01-02|    GD| 0.0|0.21003275954390174|0.20306070787008793| 0.2082897424697876|2648000.0|      0.0|         0.0|
|1962-01-02|    BP| 0.0|0.14143933090345925|0.13952797651290894|0.13952797651290894|  77440.0|      0.0|         0.0|
|1962-01-02|   MSI| 0.0| 0.7649229763450202| 0.7452535214492476| 0.7518101930618286|  65671.0|      0.0|         0.0|
+----------+------+----+-------------------+------------

In [33]:
df.printSchema()

root
 |-- Date: date (nullable = true)
 |-- Ticker: string (nullable = true)
 |-- Open: double (nullable = true)
 |-- High: double (nullable = true)
 |-- Low: double (nullable = true)
 |-- Close: double (nullable = true)
 |-- Volume: double (nullable = true)
 |-- Dividends: decimal(5,1) (nullable = true)
 |-- Stock Splits: decimal(5,1) (nullable = true)



In [34]:
# Definimos el cache
df.cache()
resumen = df.describe()
valores_nulos = df.select([
    count(when(col(c).isNull(), 1)).alias(c)
    for c in df.columns
])

In [35]:
# Disparador 1
num = df.count()

# Imprimimos el numero de columnas y registros
print(f"Número de columnas: {len(df.columns)}")
print(f"Número de registros: {num:,}\n")

Número de columnas: 9
Número de registros: 34,646,258



In [36]:
# Disparador 2
resumen_transpuesto = resumen.toPandas().set_index('summary').T
# Imprimimos el  resumen
resumen_transpuesto

summary,count,mean,stddev,min,max
Ticker,34646258,NaN,NaN,A,ZZLL
Open,34646149,1.1150488437326563E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
High,34646149,1.1150488437326608E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
Low,34646149,1.115048843732678E23,3.954734941003448E25,-8.210044423351318E25,1.5072897744279783E28
Close,34646152,1.1150487471809343E23,3.95473476978514E25,-8.210044423351318E25,1.5072897744279783E28
Volume,34646258,1339229.8683998429,1.5671698842802074E7,0.0,9.230856E9
Dividends,34646258,0.00401,1.6035252555952273,0.0,4500.0
Stock Splits,34646255,0.00039,0.1896345452067494,0.0,1000.0


In [37]:
# Disparador 3
nulos_pandas = valores_nulos.toPandas().T

# Imprimimos los valores nulos
nulos_pandas.columns = ['Valores nulos']
nulos_pandas["%"] = (nulos_pandas["Valores nulos"] / num) * 100
nulos_pandas

,Valores nulos,%
Date,0,0.000000
Ticker,0,0.000000
Open,109,0.000315
High,109,0.000315
Low,109,0.000315
Close,106,0.000306
Volume,0,0.000000
Dividends,0,0.000000
Stock Splits,3,0.000009


## Estrategia de Particionamiento

Vamos a implementar la estrategia de particionamiento descrita, utilizando las variables **Periodo económico**, **Nivel de volatilidad**, y **Volumen de transacciones**.

### Variable: Periodo Económico
Primero, definiremos una función para clasificar las fechas en 'Pre-crisis', 'Crisis' o 'Recuperación' basándonos en los eventos macroeconómicos mencionados.

In [38]:
# Definición de periodos económicos relevantes para segmentación del dataset
crisis_periods = [
    ("1987-10-01", "1988-03-31", "Crisis"), # Lunes negro
    ("2000-03-01", "2001-11-30", "Crisis"), # Burbuja de las punto com
    ("2008-09-01", "2009-03-31", "Crisis"), # Crisis financiera
    ("2020-02-01", "2020-05-31", "Crisis"), # Pandemia del Covid
    ("2022-02-01", "2023-01-31", "Crisis")  # Guerra Rusia vs Ucrania (fin arbitrario para el ejemplo)
]

# Definir periodos de recuperación (aproximados)
recovery_periods = [
    ("1988-04-01", "2000-02-29", "Recuperación"), # Después de Lunes negro, antes de Dot-com
    ("2001-12-01", "2008-08-31", "Recuperación"), # Después de Dot-com, antes de Crisis financiera
    ("2009-04-01", "2020-01-31", "Recuperación"), # Después de Crisis financiera, antes de Covid
    ("2020-06-01", "2022-01-31", "Recuperación"), # Después de Covid, antes de Guerra
    ("2023-02-01", "2024-12-31", "Recuperación")  # Después de Guerra (arbitrario para el futuro)
]

# Inicializar la columna 'Periodo_Economico' con 'Pre-crisis' como valor por defecto
df_partitioned = df.withColumn("Periodo_Economico", lit("Pre-crisis"))

# Aplicar las condiciones para Periodos de Crisis
for start_date_str, end_date_str, period_type in crisis_periods:
    start_date = to_date(lit(start_date_str))
    end_date = to_date(lit(end_date_str))
    df_partitioned = df_partitioned.withColumn("Periodo_Economico",
                                                when((col("Date") >= start_date) & (col("Date") <= end_date), lit(period_type))
                                                .otherwise(col("Periodo_Economico")))

# Aplicar las condiciones para Periodos de Recuperación
for start_date_str, end_date_str, period_type in recovery_periods:
    start_date = to_date(lit(start_date_str))
    end_date = to_date(lit(end_date_str))
    df_partitioned = df_partitioned.withColumn("Periodo_Economico",
                                                when((col("Date") >= start_date) & (col("Date") <= end_date), lit(period_type))
                                                .otherwise(col("Periodo_Economico")))

# Mostrar la distribución de los periodos económicos
df_partitioned.groupBy("Periodo_Economico").count().show()

+-----------------+--------+
|Periodo_Economico|   count|
+-----------------+--------+
|       Pre-crisis| 1560048|
|           Crisis| 4206011|
|     Recuperación|28880199|
+-----------------+--------+



### Variable: Nivel de Volatilidad

Se calcula la volatilidad diaria como la variación porcentual entre los precios máximo (*High*) y mínimo (*Low*), clasificando posteriormente los registros en niveles de volatilidad: *Baja*, *Media* y *Alta*.

In [39]:
# Mostrar la distribución de los niveles de volatilidad
df_partitioned = df_partitioned.withColumn(
    "Volatilidad_Diaria_Pct",
    expr("try_divide((High - Low) * 100, Low)")
)

df_partitioned = df_partitioned.withColumn(
    "Nivel_Volatilidad",
    when(col("Volatilidad_Diaria_Pct").isNull(), lit("Sin dato"))
    .when(col("Volatilidad_Diaria_Pct") < 2, lit("Baja"))
    .when((col("Volatilidad_Diaria_Pct") >= 2) & (col("Volatilidad_Diaria_Pct") <= 5), lit("Media"))
    .otherwise(lit("Alta"))
)

df_partitioned.groupBy("Nivel_Volatilidad").count().show()

+-----------------+--------+
|Nivel_Volatilidad|   count|
+-----------------+--------+
|             Alta| 7621183|
|            Media|10781440|
|         Sin dato|    1059|
|             Baja|16242576|
+-----------------+--------+



### Variable: Volumen de Transacciones

Se calculan percentiles sobre la variable de volumen de transacciones para clasificar los registros en niveles relativos de volumen de operación: Bajo, Medio y Alto.

In [40]:
# Calcular los percentiles para la columna 'Volume'
# Usamos approx_percentile para DataFrames grandes
volume_percentiles = df_partitioned.approxQuantile("Volume", [0.33, 0.66], 0.01)
p33 = volume_percentiles[0]
p66 = volume_percentiles[1]

print(f"Percentil 33 de Volumen: {p33:,.2f}")
print(f"Percentil 66 de Volumen: {p66:,.2f}")

df_partitioned = df_partitioned.withColumn("Volumen_Transacciones",
                                           when(col("Volume") <= p33, lit("Bajo"))
                                           .when((col("Volume") > p33) & (col("Volume") <= p66), lit("Medio"))
                                           .otherwise(lit("Alto")))

# Mostrar la distribución de los volúmenes de transacciones
df_partitioned.groupBy("Volumen_Transacciones").count().show()

Percentil 33 de Volumen: 9,200.00
Percentil 66 de Volumen: 200,100.00
+---------------------+--------+
|Volumen_Transacciones|   count|
+---------------------+--------+
|                Medio|11418009|
|                 Alto|11869581|
|                 Bajo|11358668|
+---------------------+--------+



### Combinaciones de estratos generadas

A partir de las tres variables de segmentación seleccionadas se generan combinaciones de estratos para representar subconjuntos homogéneos del dataset.

Dado que cada variable posee tres categorías, el número máximo teórico de combinaciones posibles es: 3 × 3 × 3 = 27 estratos.

Sin embargo, el número real dependerá de las combinaciones efectivamente presentes en los datos históricos analizados.

In [41]:
total_registros = df_partitioned.count()

df_estratos = df_partitioned.groupBy(
    "Periodo_Economico",
    "Nivel_Volatilidad",
    "Volumen_Transacciones"
).count()

df_estratos = df_estratos.withColumn(
    "Porcentaje",
    round((col("count") / lit(total_registros)) * 100, 4)
)

df_estratos.orderBy(col("Porcentaje").desc()).show(30, truncate=False)

+-----------------+-----------------+---------------------+-------+----------+
|Periodo_Economico|Nivel_Volatilidad|Volumen_Transacciones|count  |Porcentaje|
+-----------------+-----------------+---------------------+-------+----------+
|Recuperación     |Baja             |Bajo                 |6621937|19.113    |
|Recuperación     |Media            |Alto                 |4390511|12.6724   |
|Recuperación     |Baja             |Medio                |3841333|11.0873   |
|Recuperación     |Baja             |Alto                 |3473234|10.0248   |
|Recuperación     |Media            |Medio                |3354626|9.6825    |
|Recuperación     |Alta             |Medio                |2256682|6.5135    |
|Recuperación     |Alta             |Alto                 |2025171|5.8453    |
|Recuperación     |Alta             |Bajo                 |1656268|4.7805    |
|Recuperación     |Media            |Bajo                 |1259405|3.635     |
|Crisis           |Baja             |Bajo           

### Combinación de Particiones
Ahora, podemos ver cómo se combinan estas variables para formar las particiones. Mostraremos los primeros registros del DataFrame con las nuevas columnas de particionamiento.

In [42]:
df_partitioned.select("Date", "Ticker", "Volumen_Transacciones", "Nivel_Volatilidad", "Periodo_Economico").show(10)

# Opcionalmente, puedes contar las ocurrencias de una combinación de particiones específica
df_partitioned.groupBy("Periodo_Economico", "Nivel_Volatilidad", "Volumen_Transacciones").count().show(10, truncate=False)

+----------+------+---------------------+-----------------+-----------------+
|      Date|Ticker|Volumen_Transacciones|Nivel_Volatilidad|Periodo_Economico|
+----------+------+---------------------+-----------------+-----------------+
|1962-01-02|    ED|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   CVX|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    GD|                 Alto|            Media|       Pre-crisis|
|1962-01-02|    BP|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   MSI|                Medio|            Media|       Pre-crisis|
|1962-01-02|   HON|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    FL|                Medio|             Baja|       Pre-crisis|
|1962-01-02|    GT|                Medio|             Baja|       Pre-crisis|
|1962-01-02|   JNJ|                 Bajo|             Baja|       Pre-crisis|
|1962-01-02|   MMM|                 Alto|            Media|     

### Distribución porcentual de los estratos

Se calcula la proporción de ocurrencia de cada combinación generada para utilizarla como base del muestreo estratificado proporcional.

In [43]:
from pyspark.sql.functions import count, lit, round

total_registros = df_partitioned.count()

df_estratos = df_partitioned.groupBy(
    "Periodo_Economico",
    "Nivel_Volatilidad",
    "Volumen_Transacciones"
).count()

df_estratos = df_estratos.withColumn(
    "Porcentaje",
    round((col("count") / lit(total_registros)) * 100, 4)
)

df_estratos.orderBy(col("Porcentaje").desc()).show(27, truncate=False)

+-----------------+-----------------+---------------------+-------+----------+
|Periodo_Economico|Nivel_Volatilidad|Volumen_Transacciones|count  |Porcentaje|
+-----------------+-----------------+---------------------+-------+----------+
|Recuperación     |Baja             |Bajo                 |6621937|19.113    |
|Recuperación     |Media            |Alto                 |4390511|12.6724   |
|Recuperación     |Baja             |Medio                |3841333|11.0873   |
|Recuperación     |Baja             |Alto                 |3473234|10.0248   |
|Recuperación     |Media            |Medio                |3354626|9.6825    |
|Recuperación     |Alta             |Medio                |2256682|6.5135    |
|Recuperación     |Alta             |Alto                 |2025171|5.8453    |
|Recuperación     |Alta             |Bajo                 |1656268|4.7805    |
|Recuperación     |Media            |Bajo                 |1259405|3.635     |
|Crisis           |Baja             |Bajo           

### Técnica de muestreo por partición

Una vez construidas las particiones mediante las variables periodo económico, nivel de volatilidad y volumen de transacciones, se aplica una técnica de muestreo estratificado.

Cada combinación de estas variables representa un estrato de la población. Por ello, el muestreo estratificado permite seleccionar registros de cada partición, conservando la diversidad de escenarios presentes en el dataset.

Esta técnica es adecuada porque evita que la muestra quede dominada por los grupos con mayor número de registros y ayuda a preservar información de periodos menos frecuentes, como crisis o escenarios de alta volatilidad.

In [44]:
df_partitioned = df_partitioned.withColumn(
    "Estrato",
    concat_ws(
        "_",
        col("Periodo_Economico"),
        col("Nivel_Volatilidad"),
        col("Volumen_Transacciones")
    )
)

df_partitioned.groupBy("Estrato").count().orderBy("count", ascending=False).show(30, truncate=False)

+---------------------------+-------+
|Estrato                    |count  |
+---------------------------+-------+
|Recuperación_Baja_Bajo     |6621937|
|Recuperación_Media_Alto    |4390511|
|Recuperación_Baja_Medio    |3841333|
|Recuperación_Baja_Alto     |3473234|
|Recuperación_Media_Medio   |3354626|
|Recuperación_Alta_Medio    |2256682|
|Recuperación_Alta_Alto     |2025171|
|Recuperación_Alta_Bajo     |1656268|
|Recuperación_Media_Bajo    |1259405|
|Crisis_Baja_Bajo           |893429 |
|Crisis_Alta_Alto           |658391 |
|Crisis_Media_Alto          |631243 |
|Crisis_Alta_Medio          |542005 |
|Crisis_Media_Medio         |457087 |
|Crisis_Baja_Medio          |357435 |
|Pre-crisis_Baja_Medio      |316259 |
|Crisis_Alta_Bajo           |295893 |
|Pre-crisis_Baja_Bajo       |286782 |
|Pre-crisis_Baja_Alto       |245539 |
|Pre-crisis_Media_Medio     |223470 |
|Crisis_Baja_Alto           |206628 |
|Pre-crisis_Media_Alto      |201031 |
|Crisis_Media_Bajo          |163874 |
|Pre-crisis_

In [45]:
estratos = [row["Estrato"] for row in df_partitioned.select("Estrato").distinct().collect()]
fractions = {estrato: 0.05 for estrato in estratos}
sampled_df = df_partitioned.sampleBy(
    "Estrato",
    fractions=fractions,
    seed=42
)

print("Total población particionada:", df_partitioned.count())
print("Total muestra obtenida:", sampled_df.count())

sampled_df.groupBy("Estrato").count().orderBy("count", ascending=False).show(30, truncate=False)

Total población particionada: 34646258
Total muestra obtenida: 1732359
+---------------------------+------+
|Estrato                    |count |
+---------------------------+------+
|Recuperación_Baja_Bajo     |330629|
|Recuperación_Media_Alto    |219466|
|Recuperación_Baja_Medio    |191471|
|Recuperación_Baja_Alto     |174095|
|Recuperación_Media_Medio   |167555|
|Recuperación_Alta_Medio    |113287|
|Recuperación_Alta_Alto     |101285|
|Recuperación_Alta_Bajo     |82543 |
|Recuperación_Media_Bajo    |63015 |
|Crisis_Baja_Bajo           |44805 |
|Crisis_Alta_Alto           |33114 |
|Crisis_Media_Alto          |31635 |
|Crisis_Alta_Medio          |27391 |
|Crisis_Media_Medio         |22918 |
|Crisis_Baja_Medio          |17682 |
|Pre-crisis_Baja_Medio      |15934 |
|Crisis_Alta_Bajo           |14846 |
|Pre-crisis_Baja_Bajo       |14239 |
|Pre-crisis_Baja_Alto       |12240 |
|Pre-crisis_Media_Medio     |11113 |
|Pre-crisis_Media_Alto      |10181 |
|Crisis_Baja_Alto           |10151 |
|Cri

## 3. Preparación de los datos

Sobre la muestra estratificada **M** (`sampled_df`) obtenida en el paso anterior se aplican estrategias de corrección para dejar un conjunto **listo para los algoritmos de aprendizaje**. El pre-procesamiento contempla tres tareas:

1. **Corrección de valores nulos:** diagnóstico y eliminación de registros con nulos en columnas críticas.
2. **Identificación de valores atípicos:** detección y eliminación de registros corruptos (precios negativos o magnitudes imposibles).
3. **Transformación de tipos de datos:** conversión de tipos y generación de una variable derivada (`retorno_pct`).

El resultado es una muestra **M pre-procesada** consistente y depurada.

In [46]:
# Trabajamos sobre la muestra estratificada M obtenida en el paso de selección
M = sampled_df
M.cache()
total_M = M.count()
print(f"Registros en la muestra M: {total_M:,}\n")

# Diagnóstico de valores nulos por columna
nulos_M = M.select([count(when(col(c).isNull(), 1)).alias(c) for c in M.columns]).toPandas().T
nulos_M.columns = ["Valores nulos"]
nulos_M["%"] = (nulos_M["Valores nulos"] / total_M) * 100
nulos_M

Registros en la muestra M: 1,732,359



,Valores nulos,%
Date,0,0.000000
Ticker,0,0.000000
Open,4,0.000231
High,4,0.000231
Low,4,0.000231
Close,4,0.000231
Volume,0,0.000000
Dividends,0,0.000000
Stock Splits,0,0.000000
Periodo_Economico,0,0.000000


In [47]:
# Eliminación de registros con nulos en las columnas críticas (precios y volumen).
#     Representan < 0.001% de la muestra, por lo que su eliminación no introduce sesgo.
cols_criticas = ["Open", "High", "Low", "Close", "Volume"]
M_clean = M.dropna(subset=cols_criticas)

# Eliminación de registros 'Sin dato' en volatilidad (Low = 0 => división indefinida)
M_clean = M_clean.filter(col("Nivel_Volatilidad") != "Sin dato")

print(f"Registros tras tratar nulos : {M_clean.count():,}")
print(f"Registros eliminados        : {M.count() - M_clean.count():,}")

Registros tras tratar nulos : 1,732,307
Registros eliminados        : 52


### Identificación de valores atípicos

El resumen estadístico inicial reveló valores **imposibles**: precios negativos y magnitudes del orden de 10²³–10²⁸, producto de errores de captura en la fuente. Se aplican dos correcciones:

- **Consistencia física:** precios no negativos y `High ≥ Low` (un máximo nunca puede ser menor que el mínimo del día).
- **Tope físico sobre los precios:** la distribución de precios es **bimodal** —valores legítimos por debajo de ~10⁶ USD y registros corruptos del orden de 10²³–10²⁸, *sin valores intermedios*—. Por esa razón un **recorte por percentiles no funciona**: el percentil 99.9% cae dentro de la zona corrupta y no descarta nada. Tomando como referencia el precio por acción más alto de la historia (~700,000 USD, Berkshire Hathaway), se fija un umbral seguro en **1,000,000 USD**, que elimina únicamente los registros imposibles y conserva todas las acciones legítimas. El volumen no se filtra, pues sus valores máximos (~10⁹) son plausibles para días de alta liquidez.

In [48]:
# (a) Filtros de consistencia física: precios no negativos y High >= Low
M_clean = M_clean.filter(
    (col("Open") >= 0) & (col("High") >= 0) &
    (col("Low") >= 0) & (col("Close") >= 0) &
    (col("High") >= col("Low"))
)

# (b) Eliminación de registros corruptos mediante un TOPE FÍSICO sobre los precios.
#     La distribución de precios es BIMODAL: valores legítimos (< ~10^6 USD) y registros
#     corruptos del orden de 10^23 - 10^28, SIN valores intermedios. Con esa forma, un recorte
#     por percentiles NO funciona (el percentil 99.9% cae dentro de la zona corrupta y no filtra
#     nada). Tomando como referencia el precio por acción más alto de la historia (~700,000 USD,
#     Berkshire Hathaway), se fija un umbral físico seguro en 1,000,000 USD: descarta únicamente
#     los valores imposibles y conserva toda acción legítima.
TOPE_PRECIO = 1_000_000.0
n_antes = M_clean.count()
for c in ["Open", "High", "Low", "Close"]:
    M_clean = M_clean.filter(col(c) <= TOPE_PRECIO)

n_despues = M_clean.count()
print(f"Registros corruptos eliminados  : {n_antes - n_despues:,}")
print(f"Registros tras eliminar atípicos: {n_despues:,}\n")

# Verificación: el resumen ya no presenta magnitudes imposibles
M_clean.select("Open", "High", "Low", "Close", "Volume") \
       .describe().toPandas().set_index("summary").T

Registros corruptos eliminados  : 6,841
Registros tras eliminar atípicos: 1,720,913



summary,count,mean,stddev,min,max
Open,1720913,1360.5654674605032,23230.425080868186,0.0,1000000.0
High,1720913,1425.596651408109,24126.540482104123,1.0249949500273914E-10,1000000.0
Low,1720913,1335.5765823269758,22711.38706636596,1.0249949500273914E-10,1000000.0
Close,1720913,1378.8734264490074,23375.03674859062,1.0249949500273914E-10,1000000.0
Volume,1720913,1340778.090313107,1.4765324701510247E7,0.0,3.7554384E9


In [49]:
# Transformación de tipos de datos y generación de variable derivada
from pyspark.sql.functions import expr

M_prep = (M_clean
    # Dividends y Stock Splits venían como Decimal -> se convierten a double (requerido por MLlib)
    .withColumn("Dividends", col("Dividends").cast("double"))
    .withColumn("Stock Splits", col("Stock Splits").cast("double"))
    # Retorno intradía (%) = (Close - Open)/Open ; try_divide evita la división por cero
    .withColumn("retorno_pct", expr("try_divide((Close - Open) * 100, Open)"))
    .fillna({"retorno_pct": 0.0})
)
M_prep.cache()
print(f"Muestra M pre-procesada lista: {M_prep.count():,} registros\n")

# Verificación: el resumen ya no presenta valores imposibles (negativos ni magnitudes 10^25)
M_prep.select("Open", "High", "Low", "Close", "Volume", "retorno_pct") \
      .describe().toPandas().set_index("summary").T

Muestra M pre-procesada lista: 1,720,913 registros



summary,count,mean,stddev,min,max
Open,1720913,1360.5654674605032,23230.425080868186,0.0,1000000.0
High,1720913,1425.596651408109,24126.540482104123,1.0249949500273914E-10,1000000.0
Low,1720913,1335.5765823269758,22711.38706636596,1.0249949500273914E-10,1000000.0
Close,1720913,1378.8734264490074,23375.03674859062,1.0249949500273914E-10,1000000.0
Volume,1720913,1340778.090313107,1.4765324701510247E7,0.0,3.7554384E9
retorno_pct,1720913,0.386243413318427,105.35985505118214,-99.62162163998956,129903.46888420406


## 4. Preparación del conjunto de entrenamiento y prueba

La muestra pre-procesada **M** se divide en un conjunto de **entrenamiento (80%)** y uno de **prueba (20%)** aplicando **muestreo estratificado** sobre la variable `Estrato` (combinación de Periodo económico × Nivel de volatilidad × Volumen de transacciones).

### Justificaciones

**Muestreo estratificado:** garantiza que cada estrato esté representado en la misma proporción en entrenamiento y prueba. Un muestreo aleatorio simple podría sub-representar los estratos minoritarios, como el de "Pre-crisis_Alta_Alto", inyectando sesgo y produciendo una evaluación poco fiable. El muestreo estratificado minimiza ese riesgo de sesgo.

**División 80/20:** Al disponer de cientos de miles de registros, el 80% ofrece suficientes datos para que el árbol aprenda patrones estables, mientras que el 20% restante constituye una muestra de prueba lo bastante grande para estimar el desempeño con baja varianza. Reservar más, como por ejemplo un  30% para prueba, no aporta precisión adicional en métricas y reduce innecesariamente los datos de aprendizaje. Además 80/20 es el estándar más usado en ML.

In [50]:
from pyspark.sql.functions import monotonically_increasing_id, round as _round

# Identificador único para particionar sin solapamiento.
# Se materializa con cache()+count() para que 'id' sea estable (monotonically_increasing_id
# es no determinista si el DataFrame se recalcula).
M_model = M_prep.withColumn("id", monotonically_increasing_id())
M_model.cache()
M_model.count()

# Muestreo ESTRATIFICADO: se toma el 80% de CADA estrato para entrenamiento.
estratos_M = [r["Estrato"] for r in M_model.select("Estrato").distinct().collect()]
fracciones_train = {e: 0.8 for e in estratos_M}

train_df = M_model.sampleBy("Estrato", fractions=fracciones_train, seed=42)
# El conjunto de prueba es el complemento (registros no seleccionados), vía anti-join por id.
test_df = M_model.join(train_df.select("id"), on="id", how="left_anti")

train_df.cache(); test_df.cache()
n_train, n_test = train_df.count(), test_df.count()
print(f"Conjunto de entrenamiento: {n_train:,} ({n_train/(n_train+n_test)*100:.1f}%)")
print(f"Conjunto de prueba       : {n_test:,} ({n_test/(n_train+n_test)*100:.1f}%)\n")

# Verificación: la proporción ~80/20 se conserva en cada estrato (sin sesgo)
dist = (train_df.groupBy("Estrato").count().withColumnRenamed("count", "train")
        .join(test_df.groupBy("Estrato").count().withColumnRenamed("count", "test"), "Estrato"))
dist = dist.withColumn("%train", _round(col("train") / (col("train") + col("test")) * 100, 1))
print("Proporción por estrato (muestra de 8 estratos):")
dist.orderBy(col("train").desc()).show(8, truncate=False)

Conjunto de entrenamiento: 1,376,087 (80.0%)
Conjunto de prueba       : 344,826 (20.0%)

Proporción por estrato (muestra de 8 estratos):
+------------------------+------+-----+------+
|Estrato                 |train |test |%train|
+------------------------+------+-----+------+
|Recuperación_Baja_Bajo  |260489|65075|80.0  |
|Recuperación_Media_Alto |175387|43957|80.0  |
|Recuperación_Baja_Medio |152776|38391|79.9  |
|Recuperación_Baja_Alto  |139088|34722|80.0  |
|Recuperación_Media_Medio|133647|33758|79.8  |
|Recuperación_Alta_Medio |90423 |22806|79.9  |
|Recuperación_Alta_Alto  |81027 |20238|80.0  |
|Recuperación_Alta_Bajo  |63773 |15841|80.1  |
+------------------------+------+-----+------+
only showing top 8 rows


## 5. Construcción de modelos de aprendizaje

### Aprendizaje supervisado: Árbol de Decisión (DecisionTree)

**Problema planteado:** Voy a clasificar el nivel de volatilidad (`Baja`, `Media`, `Alta`) de un registro a partir de variables de mercado.

**Variables predictoras:** `Open`, `Close`, `Volume`, `Dividends`, `Stock Splits`, `retorno_pct` y `Periodo_Economico`.

**Prevención de fuga de información:** estoy excluyendo las variables `High`, `Low` y `Volatilidad_Diaria_Pct`, porque la etiqueta `Nivel_Volatilidad` se derivó directamente de ellas. Incluirlas haría trivial la predicción y no mediría capacidad de generalización.

El pipeline encadena: `StringIndexer` (etiqueta y variable categórica) → `VectorAssembler` (vector de características) → `DecisionTreeClassifier`. El modelo se entrena con el conjunto de entrenamiento y se evalúa con el de prueba.

In [51]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import DecisionTreeClassifier

# Etiqueta categórica -> índice numérico (la clase objetivo)
label_indexer = StringIndexer(inputCol="Nivel_Volatilidad", outputCol="label",
                              handleInvalid="skip")
# Variable categórica predictora -> índice numérico
periodo_indexer = StringIndexer(inputCol="Periodo_Economico", outputCol="Periodo_idx",
                                handleInvalid="keep")

# Variables predictoras (se excluyen High, Low y Volatilidad_Diaria_Pct: fuga de información)
feature_cols = ["Open", "Close", "Volume", "Dividends", "Stock Splits", "retorno_pct", "Periodo_idx"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

# Árbol de decisión
dt = DecisionTreeClassifier(labelCol="label", featuresCol="features", maxDepth=8, seed=42)

# Pipeline: indexado -> ensamblado de features -> entrenamiento
pipeline_dt = Pipeline(stages=[label_indexer, periodo_indexer, assembler, dt])
modelo_dt = pipeline_dt.fit(train_df)

# Correspondencia índice -> etiqueta original
print("Mapa de clases (índice -> etiqueta):")
for i, lab in enumerate(modelo_dt.stages[0].labels):
    print(f"  {i} -> {lab}")

Mapa de clases (índice -> etiqueta):
  0 -> Baja
  1 -> Media
  2 -> Alta


In [52]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Predicción sobre el conjunto de prueba
pred_dt = modelo_dt.transform(test_df)

# Métricas de desempeño
acc = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                        metricName="accuracy").evaluate(pred_dt)
f1 = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction",
                                       metricName="f1").evaluate(pred_dt)
print(f"Exactitud (accuracy) en prueba : {acc:.4f}")
print(f"F1 ponderado en prueba         : {f1:.4f}\n")

# Matriz de confusión
print("Matriz de confusión (label = real, prediction = predicho):")
pred_dt.groupBy("label", "prediction").count().orderBy("label", "prediction").show()

# Importancia de las variables predictoras
print("Importancia de variables:")
importancias = sorted(zip(feature_cols, modelo_dt.stages[-1].featureImportances.toArray()),
                      key=lambda x: x[1], reverse=True)
for nombre, imp in importancias:
    print(f"  {nombre:14s}: {imp:.4f}")

Exactitud (accuracy) en prueba : 0.7302
F1 ponderado en prueba         : 0.7232

Matriz de confusión (label = real, prediction = predicho):
+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  0.0|       0.0|145207|
|  0.0|       1.0| 15900|
|  0.0|       2.0|   192|
|  1.0|       0.0| 39115|
|  1.0|       1.0| 63054|
|  1.0|       2.0|  5703|
|  2.0|       0.0| 12097|
|  2.0|       1.0| 20014|
|  2.0|       2.0| 43544|
+-----+----------+------+

Importancia de variables:
  retorno_pct   : 0.8189
  Volume        : 0.1421
  Open          : 0.0162
  Close         : 0.0157
  Periodo_idx   : 0.0071
  Dividends     : 0.0000
  Stock Splits  : 0.0000


#### Interpretación de resultados del Árbol de Decisión

- El modelo generaliza razonablemente bien (**73% de aciertos**, F1 = 0.72) sobre datos no vistos. No es un sobreajuste trivial: la exactitud no es del 100%, lo que confirma que la exclusión de `High`, `Low` y `Volatilidad_Diaria_Pct` evitó la fuga de información.
- La importancia de variables es muy concluyente: `retorno_pct` (**0.819**) y `Volume` (**0.142**) explican el **~96%** del poder predictivo. Es coherente con la teoría financiera: los días de mayor volatilidad presentan movimientos intradía amplios y mayor volumen negociado. `Dividends` y `Stock Splits` son irrelevantes (importancia 0).
- Los errores se concentran entre clases **adyacentes** (Media↔Baja y Alta↔Media). Esto es esperable: la volatilidad es una variable continua discretizada en tres cubetas, por lo que los registros cercanos a los umbrales (2% y 5%) son intrínsecamente ambiguos. Casi no hay confusión entre extremos (solo **192** registros *Baja* clasificados como *Alta*).
- El modelo es además **robusto a los valores de escala**: tras corregir los atípicos, las métricas apenas variaron (0.7295 → 0.7302), porque los árboles parten por umbrales y no por distancias.

Podemos concluir que es un modelo **sólido, interpretable y confiable** para este problema.

### Aprendizaje no supervisado: K-Means

**Objetivo:** descubrir, sin usar etiquetas, perfiles de comportamiento de los registros bursátiles agrupándolos en clústeres homogéneos.

**Variables de agrupamiento:** `Open`, `High`, `Low`, `Close`, `Volume` y `Volatilidad_Diaria_Pct`.

K-Means se basa en distancias euclidianas, por lo que las variables se estandarizan (`StandardScaler`, media 0 y desviación 1). Sin este paso, `Volume`, cuya escala es órdenes de magnitud mayor que la de los precios, dominaría la formación de los grupos. El número de clústeres `k` se selecciona combinando el método del codo (costo WSSSE) y el coeficiente de silueta.

In [53]:
from pyspark.ml.feature import StandardScaler
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

# Variables numéricas para el agrupamiento
features_km = ["Open", "High", "Low", "Close", "Volume", "Volatilidad_Diaria_Pct"]
assembler_km = VectorAssembler(inputCols=features_km, outputCol="features_raw", handleInvalid="skip")

# Estandarización: K-Means usa distancias euclidianas, por lo que se normaliza media 0 y desviación 1 para que 'Volume' (escala mucho mayor) no domine.
scaler = StandardScaler(inputCol="features_raw", outputCol="features_scaled",
                        withMean=True, withStd=True)

prep_km = Pipeline(stages=[assembler_km, scaler]).fit(train_df)
train_km = prep_km.transform(train_df).cache()

# Selección de k mediante el coeficiente de silueta y el costo (método del codo)
evaluator_km = ClusteringEvaluator(featuresCol="features_scaled", metricName="silhouette",
                                   distanceMeasure="squaredEuclidean")

print(f"{'k':>2} | {'silueta':>9} | {'costo (WSSSE)':>16}")
print("-" * 35)
for k in range(2, 7):
    km_tmp = KMeans(featuresCol="features_scaled", k=k, seed=42)
    m_tmp = km_tmp.fit(train_km)
    sil = evaluator_km.evaluate(m_tmp.transform(train_km))
    print(f"{k:>2} | {sil:>9.4f} | {m_tmp.summary.trainingCost:>16,.1f}")

 k |   silueta |    costo (WSSSE)
-----------------------------------
 2 |    0.9975 |      4,041,544.7
 3 |    0.9979 |      2,742,977.1
 4 |    0.9958 |      2,023,241.5
 5 |    0.9909 |      1,783,775.2
 6 |    0.9950 |        933,008.2


In [54]:
# Entrenamiento final con el k elegido a partir de la celda anterior (silueta / codo)
K_OPT = 3
kmeans = KMeans(featuresCol="features_scaled", k=K_OPT, seed=42)
modelo_km = kmeans.fit(train_km)

# Evaluación sobre el conjunto de prueba (datos no usados al ajustar el modelo)
test_km = prep_km.transform(test_df)
pred_km = modelo_km.transform(test_km)
sil_test = evaluator_km.evaluate(pred_km)
print(f"Silueta en el conjunto de prueba (k={K_OPT}): {sil_test:.4f}\n")

# Tamaño de cada clúster
print("Tamaño de cada clúster (conjunto de prueba):")
pred_km.groupBy("prediction").count().orderBy("prediction").show()

# Perfil de los centroides: promedio de las variables ORIGINALES por clúster
from pyspark.sql.functions import mean as _mean, round as _round
print("Perfil promedio por clúster (variables originales):")
(pred_km.groupBy("prediction")
        .agg(*[_round(_mean(c), 3).alias(c) for c in features_km])
        .orderBy("prediction")
        .show(truncate=False))

Silueta en el conjunto de prueba (k=3): 0.9979

Tamaño de cada clúster (conjunto de prueba):
+----------+------+
|prediction| count|
+----------+------+
|         0|344264|
|         1|   562|
+----------+------+

Perfil promedio por clúster (variables originales):
+----------+----------+----------+----------+----------+-----------+----------------------+
|prediction|Open      |High      |Low       |Close     |Volume     |Volatilidad_Diaria_Pct|
+----------+----------+----------+----------+----------+-----------+----------------------+
|0         |534.527   |559.404   |523.171   |541.287   |1341886.777|4.65                  |
|1         |484587.402|498920.993|470843.406|483403.259|215.262    |6.841                 |
+----------+----------+----------+----------+----------+-----------+----------------------+



#### Interpretación de resultados K-Means

- Con `k=3` el modelo produce 2 clústeres de 344,264 y 562, y uno vacio con 0 registros, ademas de una silueta de 0.998. Una silueta tan cercana a 1 combinada con un clúster vacío **no es señal de éxito**, sino de un agrupamiento dominado por unos pocos puntos extremos.
- El clúster minoritario (562 registros) corresponde a acciones de precio altísimo (`Open ≈ 484,587`, tipo Berkshire Hathaway) con volumen ínfimo; el clúster mayoritario absorbe el **99.8%** restante.
- La causa ya no es la corrupción, sino la fuerte asimetría a la derecha de los precios legítimos (la mayoría vale pocos dólares y unas pocas decenas de miles). `StandardScaler` centra y escala por la desviación estándar, pero no corrige la asimetría, por lo que esas acciones de precio extremo siguen quedando muy lejos en el espacio euclidiano.
- Lección metodológica: K-Means es mucho más sensible que un árbol de decisión a la escala y a la distribución de las variables. Para obtener perfiles útiles se requiere un paso adicional de transformación (p. ej. logarítmica `log1p`) o un escalador robusto (`RobustScaler`) que comprima la cola de valores altos.

---

## 6. Conclusiones generales

1. **Flujo completo en PySpark.** Se aplicó el ciclo completo de aprendizaje automático sobre una muestra estratificada de ~1.72M de registros: preparación de datos, división estratificada 80/20 (perfectamente balanceada, todos los estratos en ~79.8–80.1%) y entrenamiento de un modelo supervisado y uno no supervisado.

2. **La preparación de datos fue determinante.** El recorte por percentiles inicial **no detectó** los registros corruptos (distribución bimodal); fue necesario un **tope físico** que eliminó 6,841 registros imposibles. Este paso resultó indispensable: sin él, ambos modelos —especialmente el no supervisado— quedaban inservibles.

3. **Modelo supervisado (DecisionTree): exitoso.** Alcanzó **73% de exactitud** y **0.72 de F1** sobre el conjunto de prueba, con errores concentrados solo entre clases adyacentes. `retorno_pct` y `Volume` concentran el ~96% del poder predictivo, en línea con la teoría financiera. Es robusto, interpretable y confiable.

4. **Modelo no supervisado (K-Means): requiere transformación adicional.** Aun con los datos depurados, el fuerte sesgo de los precios produce clústeres desbalanceados (uno vacío) y una silueta engañosamente alta. El resultado evidencia que K-Means exige un pre-procesamiento más exigente (normalización de la asimetría) que un árbol de decisión.

5. **Aprendizaje principal.** El contraste entre ambos algoritmos sobre el mismo conjunto de datos es, en sí mismo, el resultado más valioso del ejercicio: los métodos basados en distancias (K-Means) son intrínsecamente más sensibles a la calidad y la escala de los datos que los métodos basados en particiones por umbral (árboles de decisión).